# Delta Table Schemas - RAG Job Failure Monitor

This document defines the schema and structure of Delta tables used in the RAG Job Failure monitoring pipeline.

## Overview

The pipeline consists of four main notebooks:
1. **001** - Schema & Seed Setup (this cell)
2. **002** - Data Ingestion
3. **003** - Embedding & Vectorization
4. **004** - Query & Retrieval

Run this notebook **first** to set up the necessary schemas and tables.

## Tables

### 1. `jobs_run_log`
Stores Databricks job run history with failure details and embedding status.

| Column | Type | Description |
|--------|------|-------------|
| run_id | LONG | Databricks job run_id (primary key) |
| job_id | STRING | Databricks job_id |
| job_name | STRING | Human-readable job name |
| run_date | DATE | Date run was triggered (UTC) |
| status | STRING | SUCCESS or FAILED |
| cluster_id | STRING | Cluster that executed the run |
| triggered_by | STRING | Trigger source: scheduler, manual, etc. |
| traceback | STRING | Full stack trace for failed runs |
| error_message | STRING | Short extracted error message |
| error_type | STRING | Classified error type (e.g., NullPointerException) |
| duration_secs | INT | Run duration in seconds |
| embedded_at | TIMESTAMP | Set after vectorization; NULL if pending |

### 2. `error_registry`
Known error types with root causes, recommended actions, and owner contacts.

| Column | Type | Description |
|--------|------|-------------|
| error_type | STRING | Classified error type (must match jobs_run_log) |
| root_cause | STRING | Human-readable explanation of why this error occurs |
| actions | STRING | Numbered list of recommended remediation steps |
| email | STRING | Owner / on-call email to notify |
| severity | STRING | HIGH / MEDIUM / LOW |
| last_updated | TIMESTAMP | When this registry entry was last modified |

In [ ]:
# COMMAND ----------

# DBTITLE 1,Create schema and Delta tables (idempotent)

# Schema
spark.sql("CREATE SCHEMA IF NOT EXISTS rag_jobs")

# TABLE 1: jobs_run_log - stores every monitored job run
spark.sql("""
  CREATE TABLE IF NOT EXISTS rag_jobs.jobs_run_log (
    run_id        LONG       COMMENT 'Databricks job run_id (primary key together with job_id)',
    job_id        STRING     COMMENT 'Databricks job_id',
    job_name      STRING     COMMENT 'Human-readable job name',
    run_date      DATE       COMMENT 'Date the run was triggered (UTC)',
    status        STRING     COMMENT 'SUCCESS or FAILED',
    cluster_id    STRING     COMMENT 'Cluster that executed the run',
    triggered_by  STRING     COMMENT 'Trigger source: scheduler, manual, etc.',
    traceback     STRING     COMMENT 'Full stack trace for failed runs',
    error_message STRING     COMMENT 'Short extracted error message',
    error_type    STRING     COMMENT 'Classified error type (e.g. NullPointerException)',
    duration_secs INT        COMMENT 'Run duration in seconds',
    embedded_at   TIMESTAMP  COMMENT 'Set by notebook 003 after vectorising this record; NULL means pending embedding'
  )
  USING DELTA
  COMMENT 'Databricks job run history - source of truth for the RAG job failure pipeline'
""")

# TABLE 2: error_registry - known error patterns with owner contacts and recommended actions
spark.sql("""
  CREATE TABLE IF NOT EXISTS rag_jobs.error_registry (
    error_type    STRING     COMMENT 'Classified error type - must match jobs_run_log.error_type values',
    root_cause    STRING     COMMENT 'Human-readable explanation of why this error occurs',
    actions       STRING     COMMENT 'Numbered list of recommended remediation steps',
    email         STRING     COMMENT 'Owner / on-call email to notify',
    severity      STRING     COMMENT 'HIGH / MEDIUM / LOW',
    last_updated  TIMESTAMP  COMMENT 'When this registry entry was last modified'
  )
  USING DELTA
  COMMENT 'Known error types with root causes, actions, and contacts for L3 triage'
""")

print("\u2705 Schema and tables are ready.")


In [ ]:
# COMMAND ----------

# DBTITLE 1,Seed error_registry (idempotent MERGE)
# DBTITLE 1,Manage error_registry entries (add / update anytime)
# Run this cell whenever you need to add a new error type or update an existing entry.
# The MERGE is fully idempotent: known error_types are updated in place; new ones are inserted.
from datetime import datetime
from pyspark.sql import Row

error_data = [
    Row(error_type="NullPointerException",
        root_cause="A pipeline stage received a null DataFrame - usually a missing upstream table or failed read.",
        actions="1. Check upstream job succeeded. 2. Validate source table exists. 3. Add null-guard in transformation.",
        email="data-oncall@example.com", severity="HIGH", last_updated=datetime.now()),
    Row(error_type="AnalysisException",
        root_cause="Schema mismatch or column not found - usually after a source schema change.",
        actions="1. Compare source schema vs expected. 2. Run schema evolution check. 3. Update Delta table schema if needed.",
        email="data-eng@example.com", severity="HIGH", last_updated=datetime.now()),
    Row(error_type="OutOfMemoryError",
        root_cause="Spark executor ran out of memory - large shuffle or skewed join.",
        actions="1. Increase executor memory in cluster config. 2. Add salting for skewed joins. 3. Check for data volume spike.",
        email="infra-team@example.com", severity="MEDIUM", last_updated=datetime.now()),
    Row(error_type="FileNotFoundException",
        root_cause="Input file or Delta path does not exist at the expected location.",
        actions="1. Verify ADLS path exists. 2. Check upstream export job. 3. Validate storage mount.",
        email="data-oncall@example.com", severity="HIGH", last_updated=datetime.now()),
    Row(error_type="TimeoutException",
        root_cause="Job exceeded max allowed time - usually a slow query or undersized cluster.",
        actions="1. Check query plan for missing filters. 2. Review cluster size. 3. Add partitioning if full scan.",
        email="data-eng@example.com", severity="MEDIUM", last_updated=datetime.now()),
]

df_errors = spark.createDataFrame(error_data)
df_errors.createOrReplaceTempView("error_seed")

spark.sql("""
    MERGE INTO rag_jobs.error_registry AS target
    USING error_seed AS source
    ON target.error_type = source.error_type
    WHEN MATCHED THEN UPDATE SET
        root_cause   = source.root_cause,
        actions      = source.actions,
        email        = source.email,
        severity     = source.severity,
        last_updated = source.last_updated
    WHEN NOT MATCHED THEN INSERT *
""")

print(f"\u2705 error_registry seeded / refreshed: {df_errors.count()} entries.")


In [ ]:

# COMMAND ----------

# DBTITLE 1,Seed sample job failures in jobs_run_log (idempotent MERGE)
# DBTITLE 1,Seed demo job failures (skipped in production)
# DEMO DATA ONLY - notebook 002 populates jobs_run_log from real Databricks job run logs.
# This cell is guarded by a widget; it is skipped unless you explicitly enable it.
# Set the widget to 'true' when bootstrapping a fresh environment for local testing.
dbutils.widgets.dropdown("seed_demo_data", "false", ["false", "true"], "Seed demo data?")
if dbutils.widgets.get("seed_demo_data") != "true":
    print("⏭️  seed_demo_data=false - skipping demo seed. Set widget to 'true' to run.")
    dbutils.notebook.exit("skipped")
from datetime import date
from pyspark.sql.types import (
    StructType, StructField, LongType, StringType, DateType, IntegerType
)

schema = StructType([
    StructField("run_id",        LongType(),    True),
    StructField("job_id",        StringType(),  True),
    StructField("job_name",      StringType(),  True),
    StructField("run_date",      DateType(),    True),
    StructField("status",        StringType(),  True),
    StructField("cluster_id",    StringType(),  True),
    StructField("triggered_by",  StringType(),  True),
    StructField("traceback",     StringType(),  True),
    StructField("error_message", StringType(),  True),
    StructField("error_type",    StringType(),  True),
    StructField("duration_secs", IntegerType(), True),
])

sample_jobs = [
    (1001, "job_001", "customer_etl_daily",  date(2025, 6, 25), "FAILED", "cls-abc123", "scheduler",
     "Traceback (most recent call last):\n  File \"/jobs/customer_etl.py\", line 42, in transform\n"
     "    df = df.withColumn(\"revenue\", col(\"amount\") * col(\"rate\"))\n"
     "AttributeError: NullPointerException: Column 'rate' is null for 3 records",
     "NullPointerException: Column 'rate' is null", "NullPointerException", 143),

    (1002, "job_002", "inventory_sync",       date(2025, 6, 25), "FAILED", "cls-def456", "scheduler",
     "Traceback (most recent call last):\n  File \"/jobs/inventory_sync.py\", line 89\n"
     "pyspark.sql.utils.AnalysisException: Column 'warehouse_code' not found.\n"
     "Available columns: [id, location, qty]",
     "AnalysisException: Column 'warehouse_code' not found", "AnalysisException", 67),

    (1003, "job_003", "sales_aggregation",    date(2025, 6, 26), "FAILED", "cls-abc123", "manual",
     "java.lang.OutOfMemoryError: GC overhead limit exceeded\n"
     "  at org.apache.spark.shuffle.sort.SortShuffleWriter.write(SortShuffleWriter.scala:97)\n"
     "  at org.apache.spark.scheduler.ShuffleMapTask.runTask(ShuffleMapTask.scala:99)",
     "OutOfMemoryError: GC overhead limit exceeded", "OutOfMemoryError", 890),
]

df_jobs = spark.createDataFrame(sample_jobs, schema=schema)
df_jobs.createOrReplaceTempView("job_seed")

spark.sql("""
    MERGE INTO rag_jobs.jobs_run_log AS target
    USING job_seed AS source
    ON target.job_id = source.job_id AND target.run_id = source.run_id
    WHEN NOT MATCHED THEN INSERT (
        run_id, job_id, job_name, run_date, status, cluster_id,
        triggered_by, traceback, error_message, error_type, duration_secs
    )
    VALUES (
        source.run_id, source.job_id, source.job_name, source.run_date, source.status,
        source.cluster_id, source.triggered_by, source.traceback, source.error_message,
        source.error_type, source.duration_secs
    )
""")

print(f"\u2705 Sample job failures seeded: {df_jobs.count()} records.")


In [ ]:

# COMMAND ----------

# DBTITLE 1,Validate setup
# DBTITLE 1,Validate setup
print("=== error_registry ===")
spark.sql("""
    SELECT error_type, severity, email
    FROM rag_jobs.error_registry
    ORDER BY severity, error_type
""").show(truncate=False)

print("=== jobs_run_log ===")
spark.sql("""
    SELECT run_id, job_id, job_name, run_date, status, error_type, embedded_at
    FROM rag_jobs.jobs_run_log
    ORDER BY run_date DESC
""").show(truncate=False)

print("=== Failed jobs + triage details (join check) ===")
spark.sql("""
    SELECT j.job_id, j.job_name, j.run_date, j.error_type,
           e.severity, e.email, e.actions
    FROM rag_jobs.jobs_run_log j
    LEFT JOIN rag_jobs.error_registry e USING (error_type)
    WHERE j.status = 'FAILED'
    ORDER BY j.run_date DESC
""").show(truncate=False)